## **1. Mount google drive**
---

In [ ]:
# Running locally now, so the Google Drive mounting is no longer required.
# from google.colab import drive
# drive.mount('/content/gdrive')

## **2. Import the necessary libraries**
---

In [1]:
import matplotlib
import sklearn
import numpy as np
import pandas as pd
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
import tensorflow as tf
import os


from tensorflow.keras.callbacks import ModelCheckpoint,CSVLogger
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.utils import plot_model


print("Versions of key libraries")
print("---")
print("tensorflow: ", tf.__version__)
print("numpy:      ", np.__version__)
print("matplotlib: ", matplotlib.__version__)
print("sklearn:    ", sklearn.__version__)


Versions of key libraries
---
tensorflow:  2.15.1
numpy:       1.24.4
matplotlib:  3.10.9
sklearn:     1.7.1


## **3.Create a function to plot the japanese character correctly**
---

In [2]:
def grayplt(img,title=''):
    plt.axis('off')
    if np.size(img.shape) == 3:
        plt.imshow(img[:,:,0],cmap='gray',vmin=0,vmax=1)
    else:
        plt.imshow(img,cmap='gray',vmin=0,vmax=1)
    plt.title(title, fontproperties=prop)
    plt.show()

print(grayplt)

<function grayplt at 0x0000020EA3129510>


## **4. Setup matplotlib**
---

In [3]:
                                          # Setting up the font manager, so that
                                          # it can show japanese characters correctly
from matplotlib import font_manager as fm
fpath       = os.path.join(os.getcwd(), "ipam.ttf")
prop        = fm.FontProperties(fname=fpath)

plt.style.use('ggplot') 
plt.rcParams['ytick.right']     = True
plt.rcParams['ytick.labelright']= True
plt.rcParams['ytick.left']      = False
plt.rcParams['ytick.labelleft'] = False
plt.rcParams['figure.figsize']  = [7,7]   # Set the figure size to be 7 inch for (width,height)

print("Matplotlib setup completes.")

OSError: 'seaborn' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)

## **5. Prepare data for training and testing**
---
* Step 1: Load the dataset 
* Step 2: Check the shape and type of the data, plot a sample for observation
* Step 3: Convert the data into float32 and rescale the values from the range of 0\~255 into 0\~1
* Step 4: Retrieve the row size and the column size of each image
* Step 5: Reshape training and testing data to be in the form of `[samples,rows,columns,channel]`. This is required by Keras framework
* Step 6: Perform one-hot enconding on the labels
* Step 7: Retrieve the number of classes in this problem

In [ ]:
                                                                                # Step 1
trDat       = np.load('kmnist-train-imgs.npz')['arr_0']
trLbl       = np.load('kmnist-train-labels.npz')['arr_0']
tsDat       = np.load('kmnist-test-imgs.npz')['arr_0']
tsLbl       = np.load('kmnist-test-labels.npz')['arr_0']

                                                                                # Step 2
print("The shape of trDat is", trDat.shape, "and the type of trDat is", trDat.dtype)
print("The shape of tsDat is", tsDat.shape, "and the type of tsDat is", tsDat.dtype)
print("")
print("The shape of trLbl is", trLbl.shape, "and the type of trLbl is", trLbl.dtype)
print("The shape of tsLbl is", tsLbl.shape, "and the type of tsLbl is", tsLbl.dtype)
print("")
grayplt(trDat[132])

                                                                                # Step 3
trDat           = trDat.astype('float32')/255
tsDat           = tsDat.astype('float32')/255

                                                                                # Step 4
imgrows         = trDat.shape[1]
imgclms         = trDat.shape[2]

                                                                                # Step 5
trDat       = trDat.reshape(trDat.shape[0],
                            imgrows,
                            imgclms,
                            1)
tsDat       = tsDat.reshape(tsDat.shape[0],
                            imgrows,
                            imgclms,
                            1)

                                                                                # Step 6
trLbl           = to_categorical(trLbl)
tsLbl           = to_categorical(tsLbl)
                               
num_classes     = tsLbl.shape[1]                                                # Step 7

## **6. Define deep learning model (to be completed)**
___
* Step 1: Set a name for the coming model (required for saving)
* Step 2: Define the convolutional neural network model (to be completed)
* Step 3: Create models for training and testing
* Step 4: Display the summary of the model of interest 

**You may trial and error various structures (for input or number of channels or kernels sizes or pooling sizes) and see what happens to the performance**

In [ ]:
modelname   = 'wks5_5'                                                          # Step 1

                                                                                # Step 2
                                                                                # Structure chosen by the trial-and-error
                                                                                # documented in the cell above: base model
                                                                                # with padding='same' on both conv layers,
                                                                                # Dropout raised 0.2 -> 0.5, and the hidden
                                                                                # Dense layer widened 128 -> 256. Scored
                                                                                # 97.40% best val accuracy at 30 epochs
                                                                                # versus 96.49% for the base model.
def createModel():
    model       = Sequential()
    model.add(Conv2D(20,
                     (5, 5),
                     padding='same',
                     input_shape=(28, 28, 1),
                     activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(40,
                     (5, 5),
                     padding='same',
                     activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.5))
    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

    return model

                                                                                # Step 3
model       = createModel() # This is meant for training
modelGo     = createModel() # This is used for final testing

model.summary()                                                                 # Step 4

## **6b. Structure trial and error (what was tried, and what happened)**
___
The notes invite us to trial and error various structures. Instead of guessing,
each change was applied to the base model **one at a time** (single-variable
ablation), all runs sharing the same seed and the same data split, so that the
effect of each individual change is attributable. Training uses all 60,000
training images, with the 10,000 test images as the validation set.

**Round 1 — 15 epochs, nine structures**

| Structure (change vs base) | Best val accuracy |
|---|---|
| Dropout 0.2 to 0.5 | 96.52% |
| padding = 'same' | 96.39% |
| Hidden Dense 128 to 256 | 96.23% |
| Channels 20/40 to 32/64 | 96.10% |
| **Base (as in the notes)** | **96.09%** |
| Extra Dropout before the output layer | 96.03% |
| 32/64 + 3x3 + same (combined) | 95.85% |
| Kernels 5x5 to 3x3 | 95.83% |
| Add a third Conv block (80 filters @ 3x3) | 95.52% |

*Finding:* all nine structures landed inside a 1.0pp band, and the leaders were
only 0.1-0.4pp apart. That is too narrow to trust from a single short run, and
the combined model scored **below** the base model - so at 15 epochs the changes
were not composing, they were mostly noise.

**Round 2 — 30 epochs, the leaders re-tested**

| Structure | Best val accuracy |
|---|---|
| **drop 0.5 + same + Dense 256** | **97.40%** |
| drop 0.5 + same | 97.19% |
| drop 0.5 | 96.91% |
| same | 96.64% |
| Base | 96.49% |

*Finding:* over a longer horizon the round-1 ranking held up (the same two
changes lead), the gap over base widened to a meaningful **+0.91pp**, and this
time the changes **did** compose. So the final model uses the top-3 combination.

*Caveat:* every structure was still improving at 30 epochs, so these numbers are
a lower bound on what the full 60-epoch run reaches, and they come from a single
seed per structure rather than an average over several runs.

## **7. Create the callbacks to be applied during training**
---
* Step 1: Create a callback to save the model from an epoch when validation accuracy is the highest
* Step 2: Create a callback to save the training loss, training accuracy, validation loss and validation accuracy of each epoch into a csv file
* Step 3: Put the two callbacks objects into a list

In [ ]:
                                                                                # Step 1
folderpath      = ''
filepath        = folderpath + modelname + ".hdf5"
checkpoint      = ModelCheckpoint(filepath, 
                                  monitor='val_accuracy', 
                                  verbose=0, 
                                  save_best_only=True, 
                                  mode='max')

csv_logger      = CSVLogger(folderpath+modelname +'.csv')                       # Step 2
callbacks_list  = [checkpoint,csv_logger]                                       # Step 3

print("Callbacks created:")
print(callbacks_list[0])
print(callbacks_list[1])
print('')
print("Path to model:", filepath)
print("Path to log:  ", folderpath+modelname+'.csv')

## **8. Train the deep learning model**
___

In [ ]:
model.fit(trDat,                            # Training data
          trLbl,                            # Training label
          validation_data=(tsDat, tsLbl),   # Validation data and label
          epochs=60,                       # The amount of epochs to be trained
          batch_size=128,                   
          shuffle=True,                     # To shuffle the training data
          callbacks=callbacks_list)         # Callbacks to execute the checkpoints

## **9. Validate the deep learning model**
---
* Step 1: Load the trained weights and compile the model
* Step 2: Make prediction


In [ ]:
                                                                                # Step 1
modelGo.load_weights(filepath)
modelGo.compile(loss='categorical_crossentropy', 
                optimizer='adam', 
                metrics=['accuracy'])

predicts    = modelGo.predict(tsDat)                                            # Step 2
print("Prediction completes.")

## **10. Report classification metrics**
---
* Step 1: Setup the label
* Step 2: Convert label from one-hot to integer
* Step 3: Calculate the accuracy score
* Step 4: Generate classification report

In [ ]:
                                                                                # Step 1
labelname   = ['お O','き Ki','す Su','つ Tsu','な Na','は Ha','ま Ma','や Ya','れ Re','を Wo']
                                                                                # Step 2
predout     = np.argmax(predicts,axis=1)
testout     = np.argmax(tsLbl,axis=1)

testScores  = metrics.accuracy_score(testout,predout)                           # Step 3

                                                                                # Step 4
print("Best accuracy (on testing dataset): %.2f%%" % (testScores*100))
print(metrics.classification_report(testout,
                                    predout,
                                    target_names=labelname,
                                    digits=4))

## **11. Print confusion matrix**
---

In [ ]:
confusion   = metrics.confusion_matrix(testout,predout)
print(confusion)

## **12. Plot curves on validation loss and accuracy**
---

In [ ]:
records     = pd.read_csv(folderpath+modelname +'.csv')
plt.figure()
plt.subplot(211)
plt.plot(records['val_loss'], label="validation")
plt.plot(records['loss'],label="training")
plt.yticks([0.10,0.30,0.50,0.70])
plt.title('Loss value',fontsize=12)

ax          = plt.gca()
ax.set_xticklabels([])

plt.subplot(212)
plt.plot(records['val_accuracy'],label="validation")
plt.plot(records['accuracy'],label="training")
plt.yticks([0.7,0.8,0.9,1.0])
plt.title('Accuracy',fontsize=12)
ax.legend()
plt.show()

## **13. Save the model plot**
---

In [ ]:
plotpath  = folderpath+modelname+'_plot.png'
plot_model(model, 
           to_file=plotpath, 
           show_shapes=True, 
           show_layer_names=False,
           rankdir='TB')

print("Path to plot:", plotpath)